# Bloque 3 · Cómo obtener información de una web con ayuda de la IA — prácticas en Google Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/consumer-spain/curso-ia-colab/blob/main/bloque3/bloque3_practicas.ipynb)

**Esto es el Plan B.** Si en tu ordenador algo no funciona (Python, paquetes, permisos, el laboratorio…), puedes seguir aquí
**todas las prácticas del Bloque 3**, en el mismo orden que en clase.

**Cómo se usa**
- Cada bloque gris es una *celda*. Se ejecuta con el botón ▶ de su izquierda (o **Mayús + Enter**).
- Ejecuta las celdas **en orden**, de arriba abajo.
- Si Colab se desconecta o lo reinicias, vuelve a ejecutar la celda **Preparación**.
- Colab **no guarda tus cambios**. Si quieres conservarlos: *Archivo → Guardar una copia en Drive*.
- No subas aquí datos reales ni información interna: todo el material del curso es público o de prueba.
- La IA que uses es la misma que en clase: este cuaderno **no** la sustituye. Tú le pides el código, lo pegas en la
  celda indicada y lo ejecutas.

**En este bloque hay dos sitios de trabajo:**
- **Tu navegador** (Chrome, Edge…): webs, DevTools (F12), Inspector y Network. Eso no cambia: se hace igual que en clase.
- **Este cuaderno**: lo que en clase hace Python en tu equipo (el laboratorio, los programas, abrir los Excel).

| En clase | Aquí |
|---|---|
| Descomprimir el ZIP y comprobar `LAB_WEB` | Celda **Preparación** |
| `INICIAR_LAB.bat` → `http://localhost:8765` | Celda que arranca el laboratorio y te da un enlace |
| `py -3 archivo.py` / `python archivo.py` | `!python archivo.py` (o `%run archivo.py` si pide datos) |
| Guardar el script que te da la IA (VS Code, Bloc de notas) | Celda `%%writefile archivo.py` y pegar el código |
| Abrir un Excel con Excel | `ver_excel("archivo.xlsx")`, o `descargar("archivo.xlsx")` para abrirlo en tu equipo |
| Abrir en Excel un fichero descargado de una web | Descargarlo con tu navegador y `subir()` a Colab |
| F12, Inspector, Network, Win + Mayús + S, Ctrl + S | **Igual**, en tu navegador |

**Cómo se clasifica cada práctica** (verás una de estas etiquetas donde ocurre):

- 🟢 **IGUAL** — se hace exactamente igual que en clase.
- 🟡 **ADAPTADA** — se hace con otra herramienta, pero se aprende lo mismo.
- 🔴 **NO EXPORTABLE** — no se puede hacer en Colab; se dice qué hacer en su lugar.

⚠️ **Webs reales desde Colab.** Lo que ejecutas aquí sale a Internet desde **servidores de Google** (IP de centro de
datos, normalmente fuera de España), no desde tu equipo. Algunas webs pueden bloquearlo, cortar la conexión o mostrar
otro contenido. Por eso hay copias guardadas y respuestas reales de respaldo.

⚠️ **Nada de lo que obtengas desde Colab tiene valor probatorio.** No equivale a una captura hecha desde un equipo
institucional. Este bloque trata de *obtener* información; *conservarla como prueba* es el Bloque 4.

**Cuándo no mirar la solución.** Las referencias y la recuperación están en la última sección,
**«Solo si te has bloqueado»**. Úsala cuando la necesites, igual que en clase.

## Preparación

Descarga los materiales del Bloque 3 (la carpeta `01_ALUMNADO` del ZIP de clase), instala lo que falte, define unas
pequeñas herramientas (`ir_a`, `ver_excel`, `subir`, `descargar`, `abrir_laboratorio`…) y ejecuta el comprobador del
bloque (`INSTALACION/verificar_entorno_bloque3.py`).

> 🟡 **ADAPTADA** — en clase: descomprimir, `INSTALAR_DEPENDENCIAS.bat` y comprobar que existe `LAB_WEB`. Aquí lo hace esta celda.

In [ ]:
# PREPARACIÓN · ejecuta esta celda primero
import os, sys, zipfile, urllib.request, subprocess, shutil, time, socket
from pathlib import Path

ZIP_URL = "https://raw.githubusercontent.com/consumer-spain/curso-ia-colab/main/bloque3/BLOQUE3_MATERIALES.zip"
TRABAJO = Path.cwd().resolve()
RAIZ = TRABAJO / "BLOQUE_3_MATERIALES_v4"
BASE = RAIZ / "01_ALUMNADO"
ZIP_LOCAL = TRABAJO / "materiales_b3.zip"

if not BASE.exists():
    print("Descargando los materiales del Bloque 3...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_LOCAL)
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(TRABAJO)
    print("Materiales listos en:", BASE.relative_to(TRABAJO))
else:
    print("Los materiales ya estaban descargados:", BASE.relative_to(TRABAJO))

# Paquetes (los mismos que INSTALACION/requirements.txt, más xlrd para abrir el .xls del Geoportal)
for paquete, modulo in (("requests", "requests"), ("beautifulsoup4", "bs4"), ("pandas", "pandas"),
                        ("openpyxl", "openpyxl"), ("xlrd", "xlrd")):
    try:
        __import__(modulo)
    except ImportError:
        print("Instalando", paquete, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


def ir_a(carpeta):
    """Se coloca en una carpeta del material (como abrir la carpeta en el Explorador)."""
    os.chdir(BASE / carpeta if carpeta else BASE)
    print("Estás en:", Path.cwd().relative_to(TRABAJO))


def ver_excel(ruta, hoja=None, filas=(1, 15), columnas=8):
    """Muestra una hoja de Excel (.xlsx o .xls) como cuadrícula, con números de fila y letras de columna."""
    import pandas as pd
    from IPython.display import display
    from openpyxl.utils import get_column_letter
    ruta = Path(ruta)
    if not ruta.exists():
        print("⚠ No existe:", ruta.name, "· ¿has ejecutado antes el programa que lo crea, o lo has subido?")
        return
    libro = pd.ExcelFile(ruta)
    if hoja is None:
        hoja = libro.sheet_names[0]
    if hoja not in libro.sheet_names:
        print("⚠ No hay ninguna hoja", repr(hoja), "· hojas disponibles:", libro.sheet_names)
        return
    df = libro.parse(hoja, header=None, dtype=str).fillna("")
    print(f"{ruta.name} · hoja «{hoja}» · {len(df)} filas · hojas del libro: {libro.sheet_names}")
    parte = df.iloc[filas[0] - 1:filas[1], :columnas].copy()
    parte.columns = [get_column_letter(i + 1) for i in range(parte.shape[1])]
    parte.index = range(filas[0], filas[0] + len(parte))
    parte.index.name = "fila"
    with pd.option_context("display.max_colwidth", 40, "display.width", 250, "display.max_columns", 30):
        display(parte)


def subir():
    """Sube a esta carpeta un archivo de tu ordenador (por ejemplo, el que has descargado con tu navegador)."""
    try:
        from google.colab import files
    except ImportError:
        print("(Fuera de Colab) Copia el archivo a:", Path.cwd())
        return []
    nombres = list(files.upload().keys())
    print("Subido a", Path.cwd().relative_to(TRABAJO), ":", nombres)
    return nombres


def descargar(ruta):
    """Descarga un archivo a tu ordenador (para abrirlo con Excel o adjuntarlo a tu IA)."""
    ruta = Path(ruta)
    try:
        from google.colab import files
    except ImportError:
        print("(Fuera de Colab) El archivo está en:", ruta)
        return
    files.download(str(ruta))


def revelar(texto):
    """Muestra un texto con formato. Ejecútala solo cuando hayas pensado tu respuesta."""
    from IPython.display import display, Markdown
    display(Markdown(texto))


PUERTO = 8765
_servidor = None


def abrir_laboratorio(pagina="LAB_WEB/index.html"):
    """Equivale a INICIAR_LAB.bat: sirve las páginas del material y te da un enlace para abrirlas en tu navegador."""
    global _servidor
    if _servidor is None or _servidor.poll() is not None:
        _servidor = subprocess.Popen([sys.executable, "-m", "http.server", str(PUERTO)], cwd=BASE,
                                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(50):
            try:
                socket.create_connection(("127.0.0.1", PUERTO), timeout=0.2).close()
                break
            except OSError:
                time.sleep(0.1)
    try:
        from google.colab import output
        print("Laboratorio en marcha. Abre el enlace (se abre en otra pestaña de tu navegador):")
        output.serve_kernel_port_as_window(PUERTO, path="/" + pagina)
    except ImportError:
        print("(Fuera de Colab) Abre en tu navegador: http://localhost:%d/%s" % (PUERTO, pagina))


# --- Comprobación ---
ir_a("")
print()
print("COMPROBADOR DEL BLOQUE (INSTALACION/verificar_entorno_bloque3.py)")
r = subprocess.run([sys.executable, "INSTALACION/verificar_entorno_bloque3.py"], capture_output=True, text=True)
print(r.stdout.rstrip())
fallos = [] if r.returncode == 0 else ["paquetes de Python"]
necesarios = ["LAB_WEB/index.html", "LAB_WEB/producto.json", "GEOPORTAL/script_referencia_provincia.py",
              "GEOPORTAL/RESPALDO_API", "PROMPTS/02_PROMPT_PRACTICA_GEOPORTAL.txt",
              "MERCADONA_CONGELADO/m_respuesta_categoria.json", "BOE/boe_legislacion_consumidores.py",
              "CHULETA/CHULETA_DEVTOOLS.png"]
print()
for rel in necesarios:
    existe = (BASE / rel).exists()
    print("  " + ("✔ " if existe else "✘ ") + rel)
    if not existe:
        fallos.append(rel)
print()
print("(En Colab no hay ventanas de escritorio: el aviso sobre tkinter es normal y no afecta a este bloque.)")
print()
print("TODO LISTO" if not fallos else "HAY ALGO QUE REVISAR: " + ", ".join(fallos))

---
# 1 · DEFINIR

*(Diapositivas 4–7)*

**Ruta del bloque:** DEFINIR → DAR CONTEXTO → ELEGIR LA VÍA → RESOLVER → COMPROBAR.

## Antes de pedir nada: concretar el encargo

> 🟢 **IGUAL** — actividad de pensar; en clase, por parejas.

Elige un caso:

- **A** · ¿Cuánto cuesta hoy este producto en esta tienda online?
- **B** · ¿Qué información sobre la vivienda y sus condiciones aparece en este anuncio?
- **C** · ¿Sigue esta oferta igual dentro de una semana?

Responde a tres preguntas, **sin tecnología**:
1. ¿Qué queremos comprobar exactamente?
2. ¿Cómo lo haríamos manualmente?
3. ¿Qué resultado necesitamos al final?

Cuando lo tengas, ejecuta la celda siguiente.

In [ ]:
revelar("""
## La necesidad determina la solución

| | Qué comprobamos | Cómo, a mano | Qué resultado |
|---|---|---|---|
| **A · Precio hoy** | Un precio, ahora | Abrir la ficha y leerlo | Precio + fecha + enlace |
| **B · Anuncio de vivienda** | ¿Qué información? Aún es vago: hay que concretarlo | Abrir el anuncio y revisar sus campos | Los campos que decidamos: precio, ubicación, superficie, habitaciones… |
| **C · Oferta en 7 días** | El mismo dato, dos veces | Mirar hoy y dentro de 7 días | Dos valores con fecha |

**DEFINIR ✓ · 1/5** — sabemos qué necesitamos, cómo lo haríamos a mano y qué resultado esperamos.
Todavía no hemos decidido la técnica.
""")

---
# 2 · DAR CONTEXTO

*(Diapositivas 8–11)*

> 🟢 **IGUAL** — es una conversación con tu IA; funciona igual en cualquier sitio.

## Contexto inicial: tu primer mensaje a la IA

Ya puedes darle: **el objetivo · la URL · cómo lo harías a mano · el resultado que esperas**.
Y pedirle que, antes de programar, te ayude a diagnosticar cómo ofrece los datos la web.

**LA IA PROPONE. NOSOTROS COMPROBAMOS.**

El prompt completo es `PROMPTS/01_PROMPT_PRIMER_CONTEXTO.txt`:

In [ ]:
ir_a("PROMPTS")
!cat 01_PROMPT_PRIMER_CONTEXTO.txt

## Dar contexto tiene dos niveles

- **Contexto inicial** (antes de saber nada técnico): objetivo · URL · cómo lo haría manualmente · resultado que espero.
- **Contexto adicional** (si la IA necesita saber más): captura · HTML de la página · elemento inspeccionado · Network / Response.
  Lo aprenderemos cuando haga falta.

## No es un único prompt

*Ejemplo de conversación:*

> **TÚ:** «Quiero obtener esta información de esta web.» → **IA:** «Necesito ver X.» → **TÚ:** «¿Cómo consigo X?» →
> **IA:** «Haz esto…» → **TÚ:** obtienes X y se lo vuelves a dar → **IA:** «Con esto probaría primero esta vía.»

PREGUNTA → OBTÉN CONTEXTO → DEVUÉLVELO → SIGUE.

**«¿Qué necesitas de mí y cómo puedo conseguirlo?»**

**DAR CONTEXTO ✓ · 2/5**

---
# Caso real 1 · Carburantes · ELEGIR LA VÍA

*(Diapositivas 12–21)*

Primer caso de trabajo: de la web oficial del Geoportal a un Excel.

Vías que descubriremos hoy, de lo más sencillo a lo más complejo:
**DESCARGA → API → HTML → NETWORK → NAVEGADOR**. En cualquier nivel: ¿tiene sentido seguir? → replantear.

*Ejemplo de conversación:*
> **TÚ:** «Necesito conocer los precios actuales de las gasolineras españolas. Antes de programar, comprueba si la fuente oficial ofrece ya los datos.»
> **IA:** «El Geoportal oficial permite descargar un fichero actualizado. Probaría primero esa opción.»

**Ahora lo comprobamos.**

## «Necesito los precios actuales de las gasolineras españolas» · ¿existe ya una descarga?

> 🟢 **IGUAL** (buscar y descargar, en tu navegador) · 🟡 **ADAPTADA** (abrir el fichero: aquí lo subes a Colab y lo miras con `ver_excel`, o lo abres en tu equipo si tienes Excel).

1. **BUSCAR** — en tu navegador: https://geoportalgasolineras.es/geoportal-instalaciones/DescargarFicheros
2. **DESCARGAR** — «Información actualizada de precios».
3. **ABRIR** — ejecuta la celda de abajo y elige el fichero que acabas de descargar.
4. **COMPROBAR** — fecha · ámbito · columnas · precios.

In [ ]:
ir_a("GEOPORTAL")
subidos = subir()

In [ ]:
# Muestra el fichero que has subido (si tiene otro nombre, escríbelo entre las comillas)
ir_a("GEOPORTAL")
ver_excel(subidos[0] if subidos else "nombre_del_fichero.xls", filas=(1, 15), columnas=8)

**Comprueba tú:** ¿qué fecha tiene el fichero? ¿es de toda España? ¿hay una fila por estación? ¿hay precios por carburante?

Si quieres ver más filas o columnas: `ver_excel(..., filas=(1, 40), columnas=12)`.

**Primer éxito: no programar era la solución correcta.** DESCARGA ✓

## Cambia la necesidad

*Ejemplo de conversación:*
> **TÚ:** «Ahora no necesito solo los precios actuales. Necesito varias fechas y reunirlas en un mismo archivo.»
> **IA:** «La descarga actual ya no basta. La fuente oficial permite también consultar datos históricos por fecha.»

**Comprobemos que existe.**

## ¿Qué es una API?

Una forma de pedir datos directamente a un servicio, sin tener que leerlos de la pantalla.
PERSONA → página web → ve los datos · PROGRAMA → API → recibe datos estructurados.

**Todo lo que necesitamos saber:** si una fuente ofrece una API, puede ahorrarnos mucho trabajo. No necesitamos saber cómo
está construida. Solo comprobar que existe, qué podemos pedirle y qué devuelve.

## La dirección tiene dos huecos

`.../FiltroProvincia/{FECHA}/{PROVINCIA}` · ejemplo: `.../FiltroProvincia/14-10-2026/28` → **14-10-2026** = FECHA ·
**28** = PROVINCIA DE MADRID (otras: 46 Valencia · 08 Barcelona · 41 Sevilla).

## Abre la API en tu navegador

> 🟢 **IGUAL** — se hace en tu navegador, no en Colab.

1. Abre el enlace de `GEOPORTAL/FUENTES_OFICIALES.txt` (celda de abajo).
2. Cambia **solo la fecha**. 3. Pulsa Enter. 4. Mira el principio: ¿ha cambiado la fecha?

Se ve «en bruto», y no pasa nada: lo importante es que la fecha cambió.

In [ ]:
ir_a("GEOPORTAL")
!cat FUENTES_OFICIALES.txt

**ELEGIR LA VÍA ✓ · 3/5** — la IA propuso dos veces y las dos veces lo comprobamos:
hoy → descarga del fichero actualizado · varias fechas → API oficial (fecha + provincia).

---
# Práctica Geoportal · RESOLVER

*(Diapositivas 22–28)*

## Vamos a construir algo

**Obtener los precios de las gasolineras de una provincia para 2 o 3 fechas, en un único Excel.** ¿Cómo empezaríais?

**Ya sabemos:** que existe una API oficial · su dirección y qué cambia en ella · qué resultado queremos.
**Contexto para la IA:** objetivo · fechas · provincia · salida: un Excel · URL oficial + documentación.
Le damos la vía ya comprobada: la IA resuelve cómo usarla, no inventa otra fuente.

**Criterios mínimos:** un único archivo Excel · una fila por estación y fecha · una columna con la fecha ·
todas las fechas pedidas, o un aviso de cuál falló · los precios no están sistemáticamente vacíos · solo la provincia pedida.

## Ahora vosotros · 10–12 min

> 🟡 **ADAPTADA** — el código de la IA se pega en una celda `%%writefile` y se ejecuta con `!python`.
> La API se consulta desde los servidores de Google: puede tardar, fallar o cortar la conexión (ver «Un fallo real»).

1. Abre tu IA y copia el prompt base (celda de abajo).
2. Completa fechas (2 o 3 de la semana pasada) y provincia.
3. **Añade al final:** *«Lo voy a ejecutar en Google Colab con `!python mi_geoportal.py`.»*
4. Deja que la IA pregunte. Genera el script.

In [ ]:
ir_a("PROMPTS")
!cat 02_PROMPT_PRACTICA_GEOPORTAL.txt

5. Ejecuta la primera celda de abajo (te coloca en `GEOPORTAL`). Pega el código de la IA **debajo de la primera línea**
   de la segunda celda (deja `%%writefile mi_geoportal.py` arriba) y ejecútala: así se guarda el archivo.

In [ ]:
ir_a("GEOPORTAL")

In [ ]:
%%writefile mi_geoportal.py
# Pega aquí, debajo de esta línea, el código que te ha dado la IA

6. Ejecútalo. Si **falla**: copia el código y el error **completo** y usa `PROMPTS/05_PROMPT_CORREGIR_ERROR.txt`
   (para cambiar el código, vuelve a pegarlo corregido en la celda `%%writefile` y ejecútala otra vez).

In [ ]:
ir_a("GEOPORTAL")
!python mi_geoportal.py

Si tu programa **te pide datos** (fechas, provincia…), ejecútalo con esta otra celda: aparecerá un cuadro de texto.
*(Si lo lanzaste con `!python` y la celda se queda esperando, detenla con ■ y usa esta.)*

In [ ]:
ir_a("GEOPORTAL")
%run mi_geoportal.py

In [ ]:
ir_a("PROMPTS")
!cat 05_PROMPT_CORREGIR_ERROR.txt

## ¿Dónde estás?

¿Tienes ya código? · ¿Lo has guardado (celda `%%writefile`)? · ¿Has hecho una primera ejecución? ·
¿Tienes un error que puedas copiar? · ¿Ya tienes el archivo?

**¿No consigues avanzar?** → AYUDA (la dirección está en `FUENTES_OFICIALES.txt`; el prompt de corrección, arriba) →
RECUPERACIÓN (sección **«Solo si te has bloqueado»**, al final).

## Un fallo real

Preparando esta práctica apareció un error de conexión. La IA respondió que no venía del programa ni de la API, sino de un
proxy de la red. **Eso es una hipótesis de la IA:** el error *apunta* a un problema de red/proxy. Se comprueba con la
recuperación (otra ejecución, con el respaldo o desde otra red).

Error → copiar entero → qué intentábamos → código + error + contexto → hipótesis de la IA → comprobamos con la
recuperación → nueva ejecución.

**También los diagnósticos de la IA se comprueban.**

*En Colab te puede pasar algo parecido de verdad: la conexión sale desde servidores de Google y el servicio puede
cortarla. Haz lo mismo: copia el error completo, pide un diagnóstico y compruébalo con la recuperación.*

## Una ejecución que produce resultado

> 🟡 **ADAPTADA** — en clase abres el Excel; aquí lo miras con `ver_excel` (o lo descargas).

¿Qué Excel ha generado tu programa?

In [ ]:
ir_a("GEOPORTAL")
!ls -la *.xlsx

In [ ]:
# Escribe entre las comillas el nombre de tu Excel
ir_a("GEOPORTAL")
ver_excel("nombre_de_tu_excel.xlsx", filas=(1, 12), columnas=8)

Ya tenemos **herramienta · ejecución · archivo**. **RESOLVER ✓ · 4/5**

↓ **¿Hemos terminado?**

---
# COMPROBAR

*(Diapositivas 29–33)*

> 🟡 **ADAPTADA** — en clase se proyecta un resultado; aquí lo prepara la celda siguiente y lo miras con `ver_excel`.

Vamos a revisar **un resultado** de esta misma práctica (provincia 28, tres fechas pedidas).

In [ ]:
#@title Prepara el resultado que vamos a revisar (ejecútala; no hace falta abrirla)
import pandas as pd, tempfile
_tmp = Path(tempfile.mkdtemp())
subprocess.run([sys.executable, str(BASE / "GEOPORTAL/script_referencia_provincia.py"), "--respaldo",
                str(BASE / "GEOPORTAL/RESPALDO_API")], cwd=_tmp, capture_output=True, text=True)
_f = next(_tmp.glob("*.xlsx"))
_d, _c = pd.read_excel(_f, "Datos"), pd.read_excel(_f, "Control")
_d, _c = _d[_d["Fecha consulta"] != "13-10-2026"], _c[_c["Fecha"] != "13-10-2026"]
(BASE / "COMPROBAR").mkdir(exist_ok=True)
RESULTADO_A_REVISAR = BASE / "COMPROBAR/resultado_provincia28_tres_fechas.xlsx"
with pd.ExcelWriter(RESULTADO_A_REVISAR, engine="openpyxl") as _xw:
    _d.to_excel(_xw, sheet_name="Datos", index=False)
    _c.to_excel(_xw, sheet_name="Control", index=False)
print("Resultado listo:", RESULTADO_A_REVISAR.relative_to(TRABAJO))

In [ ]:
ver_excel(RESULTADO_A_REVISAR, "Datos", filas=(1, 10), columnas=6)

Abre bien. Tiene cabeceras. Tiene precios.

## ¿Lo daríais por bueno? ¿Qué comprobaríais?

> 🟢 **IGUAL** — pregunta abierta. Piénsalo (o escríbelo) **antes de seguir**.

In [ ]:
revelar("""
## Ejemplos de comprobaciones concretas

| Encargo | ↔ | Comprobación |
|---|---|---|
| **Pedimos 3 fechas** | ↔ | Deben aparecer 3 fechas distintas. |
| **Pedimos Madrid** | ↔ | Ninguna fila debería ser de otra provincia. |
| **Pedimos precios** | ↔ | No deberían estar sistemáticamente vacíos. |
| **Volumen** | ↔ | El número de filas de cada día debería ser parecido. |
| **Comprobación manual** | ↔ | Elegimos 2–3 estaciones y contrastamos fecha y precio con la fuente. |

*Cada cosa que pedimos se convierte en algo que podemos comprobar.*
""")

Haz tú la primera comprobación. Mira la hoja «Control» del mismo Excel: **¿cuántas fechas hay?**

In [ ]:
ver_excel(RESULTADO_A_REVISAR, "Control", filas=(1, 6), columnas=3)

Cuando tengas tu respuesta, ejecuta la celda siguiente.

In [ ]:
revelar("""
## Falta una fecha

**Pedimos 3 fechas → solo hay 2 → falta una fecha.** Sin ningún error en pantalla. El archivo se abría bien y tenía precios.

*Es un ejemplo didáctico: es un resultado real (respuestas guardadas de la API) al que le quitamos una fecha a propósito
para practicar la detección.*

## Tres tipos de fallo · Recordad el Bloque 2

1. **Error visible** — el programa se detiene.
2. **Resultado incompleto** — termina… pero falta algo. ← *lo que acabamos de ver*
3. **Resultado plausible pero incorrecto** — parece correcto… y no lo es.
""")

Ahora aplica las mismas comprobaciones **a tu propio Excel** (el de la práctica): fechas, provincia, precios, volumen y
2–3 estaciones contrastadas con la fuente.

In [ ]:
# Escribe entre las comillas el nombre de tu Excel y, si tiene otra hoja de control, su nombre
ir_a("GEOPORTAL")
ver_excel("nombre_de_tu_excel.xlsx", hoja=None, filas=(1, 12), columnas=8)

**COMPROBAR ✓ · GEOPORTAL 5/5 · Primer recorrido completo.**

Ya conocemos el método. Hasta ahora la fuente nos lo había puesto bastante fácil: una descarga y después una API oficial.
**¿Qué ocurre cuando la información está más escondida?**

---
# Ampliar el contexto · Laboratorio

*(Diapositivas 34–45)*

**No empezamos de cero: ampliamos el contexto cuando hace falta.** Si descubrimos algo nuevo, podemos volver a un paso
anterior. Contexto adicional: CAPTURA → HTML → INSPECTOR → NETWORK.

## Laboratorio · Cómo dar más contexto a la IA · 12–15 min

Una sola práctica en seis pasos: 1 Arrancar · 2 Captura y HTML · 3 DevTools · 4 Cambiar el precio · 5 Inspector · 6 Network.

### Paso 1 · Arrancar

> 🟡 **ADAPTADA** — en clase, `INICIAR_LAB.bat` abre `http://localhost:8765`: *localhost* es «este mismo ordenador».
> Aquí la tienda la sirve **la máquina de Colab** y la abres con un enlace en otra pestaña de tu navegador
> (la dirección no será `localhost`). Todo lo demás del laboratorio se hace **igual**, en tu navegador.

Ejecuta la celda y abre el enlace. Tienes que ver **TIENDA DEMO** con las «Zapatillas Demo» a 49,99 €.

In [ ]:
abrir_laboratorio("LAB_WEB/index.html")

### Paso 2 · Enseñar a la IA lo que ves

> 🟢 **IGUAL** — en tu navegador, sobre la tienda.

- **CAPTURA:** Win + Mayús + S (o Impr Pant / Herramienta Recortes) → *esto es lo que veo*.
- **HTML:** Ctrl + S → Tipo «Página web, solo HTML» → *una copia de la página que puedo proporcionar a la IA*.
  Se guarda en tu ordenador, como en clase.

### Paso 3 · Abre las herramientas del navegador

> 🟢 **IGUAL** — F12 (o Fn + F12) · Ctrl + Mayús + I · botón derecho → Inspeccionar. Hoy solo: **Elements** y **Network**
> (en español: Elementos y Red).

Chuleta: `CHULETA/CHULETA_DEVTOOLS.png` (celda de abajo).

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(BASE / "CHULETA/CHULETA_DEVTOOLS.png"), width=900))

### Paso 4 · Cambia el precio… en tu navegador

> 🟢 **IGUAL**

1. Botón derecho sobre el precio → Inspeccionar. 2. Doble clic sobre `49,99 €`. 3. Escribe `9,99 €`. 4. Enter.

**¿Hemos bajado el precio de la tienda?** Piénsalo y pulsa **F5**.
NO: solo ha cambiado lo que muestra tu navegador. F5 → vuelve el valor original: 49,99 €.

*Bola extra:* ¿te atreves con una web conocida? Abre Google, inspecciona un botón y cambia su texto. Luego, F5.

### Paso 5 · Señalar un dato con precisión (Inspector)

> 🟢 **IGUAL**

QUIERO ESTE DATO → LO LOCALIZO → INSPECCIONO → **COMPRUEBO QUE ES EL FRAGMENTO CORRECTO** → LO COPIO (Copy → Copy element)
→ LO PEGO EN LA IA, por ejemplo con: *«Este fragmento corresponde al precio de un producto. ¿Qué necesitas para
localizar todos los productos de esta página?»*

### Paso 6 · Network

> 🟢 **IGUAL** — en tu navegador. Las peticiones mostrarán la dirección de Colab en lugar de `localhost`.

Un instante después de recargar, el precio ya está… la disponibilidad todavía no. **¿De dónde llega?**

F12 → **Network** (Red) → recarga (**F5**) → **Fetch/XHR** si hay muchas líneas → clic en la petición `producto.json` →
**Response** (Respuesta). ¿Dónde está la disponibilidad?

*Si el enlace del laboratorio ha dejado de funcionar, vuelve a ejecutar la celda del Paso 1.*

## ¿Qué aporta cada cosa?

| | Aporta | Límite |
|---|---|---|
| URL + explicación | Punto de partida | La IA puede no ver lo mismo |
| Captura | Qué vemos y qué dato importa | No explica la estructura |
| HTML guardado | El HTML guardado de esa página | Puede faltar información que la página carga después |
| Elemento inspeccionado | Señala un dato con precisión | Solo una pequeña parte |
| Network / Response | Los datos que la página recibe aparte | Más técnico; no siempre hace falta |

## ¿Cómo nos puede ayudar la IA?

Identificar qué información falta · decirnos qué material necesita · guiarnos para obtenerlo · decirnos qué petición
mirar · interpretar una Response · escribir una pequeña herramienta de diagnóstico.

**Si no sabes qué mirar, no adivines. Pregunta a la IA qué necesita y cómo puedes conseguirlo.**

*API oficial (publicada y documentada para que la usemos) ≠ petición que observamos en Network (la usa la propia página;
puede cambiar sin aviso).*

---
# Caso real 2 · Comercio

*(Diapositivas 46–63)*

Una web que no nos lo pone tan fácil: seguimiento de precios en un supermercado online (caso real, tienda.mercadona.es).

- **DEFINIR:** todos los productos, con precio y tamaño, de todas las categorías, en un Excel.
- **DAR CONTEXTO:** dos páginas guardadas (la de categorías y la página donde nosotros vemos los productos) + objetivo + salida.
- **La IA pregunta** («¿Cómo quieres conservar el histórico entre ejecuciones?») **y entrega una primera solución.**
- **Primer intento:** encuentra las categorías… y **0 productos**. Sin ningún error.

**¿Qué sabe ya la IA?** el objetivo, las categorías, los dos HTML. **¿Qué le falta?** dónde están realmente los productos.

**La ruta tiene un orden, pero podemos volver a un paso anterior.** ↺ Volvemos a DAR CONTEXTO.

**No os lo cuento. Investigadlo.**

## Antes de la misión · Network en una web real: empieza limpio

> 🟢 **IGUAL** — en tu navegador.

1. Abre **Network**. 2. Pulsa **Clear network log** (el círculo tachado). 3. Recarga la página. 4. Reproduce la acción que
hace aparecer los datos. 5. Si hay demasiadas: **Fetch/XHR**.

## Misión: investiga la web vosotros · 10–12 min

> 🟢 **IGUAL** (plan A, en tu navegador: https://tienda.mercadona.es, código postal 28001) ·
> 🟡 **ADAPTADA** (plan B, con el material guardado, celdas de abajo).
> No intentes descargar la web desde Colab: saldría desde servidores de Google y puede bloquearse o responder distinto.

1. **PÁGINA** — encuentra las categorías · abre una categoría.
2. **HTML** — guarda la página de categorías · guarda una categoría (Ctrl + S → solo HTML).
3. **INSPECTOR** — inspecciona una tarjeta · copia el fragmento del precio.
4. **NETWORK** — abre Network y recarga · localiza la petición · abre Response · encuentra nombre y precio.

### Plan B · si la web cambia, pide código postal, bloquea o no carga

Material real guardado (`MERCADONA_CONGELADO`): la página de la categoría 112 «Aceite, vinagre y sal», una tarjeta, su
fragmento HTML, la respuesta que recibe la página y capturas.

In [ ]:
ir_a("MERCADONA_CONGELADO")
!cat LEEME.txt

In [ ]:
# 1 · PÁGINA y 2 · HTML: abre la página guardada en tu navegador (con F12 puedes inspeccionarla)
abrir_laboratorio("MERCADONA_CONGELADO/MERCADONA_categoria_112_guardada.html")

In [ ]:
# Capturas reales: el menú de categorías y una tarjeta de producto
display(Image(filename=str(BASE / "MERCADONA_CONGELADO/m_menu_categorias.png"), width=300))
display(Image(filename=str(BASE / "MERCADONA_CONGELADO/m_tarjeta_producto.png"), width=250))

In [ ]:
# 3 · INSPECTOR: el fragmento de esa tarjeta (lo que copiarías con Copy element)
!cat m_tarjeta_fragmento.html

In [ ]:
# 4 · NETWORK: la petición «112» tal como se ve en DevTools (captura real)…
display(Image(filename=str(BASE / "MERCADONA_CONGELADO/network_devtools.png"), width=650))

In [ ]:
# …y el principio de su Response, tal como llega a la página. Busca tú el nombre y el precio de la tarjeta.
texto = (BASE / "MERCADONA_CONGELADO/m_respuesta_categoria.json").read_text(encoding="utf-8")
print(texto[:6000])

*Para buscar dentro de la respuesta completa, ábrela en el panel 📁 (doble clic) o usa
`!grep -o '"display_name": "[^"]*"' m_respuesta_categoria.json | head`.*

## ¿Qué encontrasteis?

> 🟢 **IGUAL** — puesta en común.

¿Estaban los productos en el HTML? · ¿Qué encontrasteis con el Inspector? · ¿Nombre y precio en Response, ordenados? ·
¿Qué le daríamos a la IA? · ¿Qué le pediríamos que hiciera con ello? · **¿Cómo nos puede ayudar ahora la IA?**

## ¿Siguen saliendo demasiadas peticiones?

*Ejemplo de conversación:*
> **TÚ:** «Estoy intentando localizar qué petición contiene productos y precios. Esta es una captura de las peticiones que
> aparecen. ¿Cuáles revisarías primero y por qué?»
> **IA:** «Probaría primero las de tipo fetch cuyo nombre recuerde a los datos, como la que lleva el número de la categoría
> (112), y las de mayor tamaño. Abre una y enséñame su Response.»

**La IA también puede guiarnos dentro de DevTools.**

## Lo que la IA puede pedirte

1. Una captura (Win + Mayús + S). 2. La Response (clic derecho en la petición → Copy → Copy response).
3. La URL de la petición, si aporta (Copy → Copy URL).

**Si necesita algo más, la IA puede explicarte cómo obtenerlo.**
*Opciones avanzadas · no hace falta memorizarlas: Copy as cURL · Copy all listed as HAR (sanitized). Revisa siempre lo
que compartes: «sanitized» no garantiza que un archivo sea seguro para compartir en cualquier contexto.*

## Tarjeta ↔ petición ↔ Response

Lo que encontramos nosotros: la tarjeta (Aceite de oliva 0,4º Hacendado, garrafa 5 L, 17,25 €), la petición «112» (la
categoría que estamos viendo) y su Response, con el mismo producto y los mismos datos, estructurados.

**Ahora sí** sabemos de dónde llegan los productos: la respuesta estructurada descubierta en Network.
**ELEGIR LA VÍA ✓ · 3/5**

## No adivines: diagnostica

> 🟢 **IGUAL** — en clase es un caso real que se comenta; no hay práctica de programación.

Tras los 0 productos, se copió la salida y se preguntó a la IA «¿Puedo hacer algo? ¿O sabes qué está pasando?».
La IA propuso primero un pequeño **programa de diagnóstico** (resumen fiel: no hay captura legible de esa respuesta).
También puedes pedirlo tú: *«Antes de cambiar el programa principal, hazme un pequeño programa que me permita ver cómo
está estructurada realmente la respuesta.»*

EJECUTA el diagnóstico → COPIA el resultado → VUELVE A LA IA.
0 productos → diagnóstico → un nivel más (`categories → products`) → corrección → 35 · 48 · 17 · 40 productos.

**Resultado: el contexto depende del objetivo.** Si solo necesitamos el dato, la respuesta estructurada que descubrimos
en Network puede bastar. **RESOLVER ✓ · 4/5**

## ¿Ahora sí funciona?

> 🟡 **ADAPTADA** — con el material guardado. **Hazlo tú**: no lo damos por bueno solo porque ahora aparecen productos.

Elegir 2–3 productos de la web · ¿coincide el nombre? · ¿coincide el precio? · ¿las categorías tienen productos? ·
¿hay precios masivamente vacíos? · ¿el Excel tiene un número de filas razonable?

Con el material guardado: compara la tarjeta (captura de arriba) con su entrada en la Response.

In [ ]:
ir_a("MERCADONA_CONGELADO")
!grep -n -m 3 -A 25 "Aceite de oliva 0,4" m_respuesta_categoria.json | head -60

**COMPROBAR ✓ · MERCADONA 5/5.** Hipótesis, fallo, más contexto, vía revisada, corrección: volver atrás forma parte del método.

---
# Demostración · Vivienda

*(Diapositivas 64–66)*

Carburantes ✓ · Comercio ✓ · **Vivienda**. Hasta ahora hemos podido obtener los datos sin manejar la página como una
persona. **¿Qué pasa cuando hay que interactuar?**

> 🔴 **NO EXPORTABLE** — en clase es una **demostración en vivo** del docente (navegador automatizado sobre una web de
> vivienda). No es una práctica del alumnado. En Colab un navegador automatizado solo funciona sin ventana, su
> instalación es pesada y la web real se visitaría desde servidores de Google. **Qué hacer:** sigue la explicación.

NAVEGAR → ESPERAR → RECOGER → COMPROBAR.
**Difícil no significa imposible.** Pero es más frágil, más costoso, más difícil de mantener y más difícil de validar.

## Escalar es una decisión

¿Tiene sentido seguir? CAPTCHA · autenticación · medidas anti-bot · datos sensibles · volumen desproporcionado ·
impacto en el servicio · condiciones de acceso · hay una fuente alternativa · no podemos comprobar el resultado.

**Poder técnicamente ≠ tener que hacerlo.**

---
# El mapa, ahora completo

*(Diapositivas 68–71)*

| Vía | Dónde la vimos |
|---|---|
| ✓ DESCARGA | Carburantes: precios de hoy |
| ✓ API | Carburantes: histórico |
| ✓ HTML | Comercio: Mercadona |
| ✓ NETWORK | Comercio: Mercadona |
| ✓ NAVEGADOR | Vivienda: demostración |

En cualquier nivel: ¿tiene sentido seguir? → replantear.
**Empieza por la izquierda. Escala solo si hace falta.**

## Ahora usad el mapa

> 🟢 **IGUAL** — en clase, por parejas. ¿Por dónde empezaríais?

1. La Administración publica un fichero actualizado cada día.
2. Hay una dirección oficial que devuelve los datos de una fecha.
3. El dato está escrito en la ficha del producto.
4. Al aplicar un filtro, ves en Network que el navegador hace una nueva petición y recibe los datos de la tabla.
5. El dato solo aparece tras navegar, aceptar avisos y bajar por la página, y no hemos encontrado una descarga, API o
   petición directa reutilizable.
6. Hace falta tu usuario personal, hay CAPTCHA y no podrías comprobar el resultado.

In [ ]:
revelar("""
**Corrección:** 1 → DESCARGA · 2 → API · 3 → HTML · 4 → NETWORK · 5 → NAVEGADOR · 6 → **REPLANTEAR**

*En la realidad no siempre será tan limpio.*

**Nuestro repertorio ya está completo:** descarga · API · HTML / Inspector · Network · navegador.
Empezar por la opción más sencilla · ampliar el contexto cuando hace falta · escalar solo cuando tiene sentido.
""")

---
# Lo esencial del Bloque 3

*(Diapositiva 72)*

**Ruta completa:** Geoportal ✓ · Mercadona ✓ · **Ámbitos trabajados:** Carburantes ✓ · Comercio ✓ · Vivienda (demo) ✓

DEFINIR → DAR CONTEXTO → ELEGIR LA VÍA → RESOLVER → COMPROBAR.

Que termine sin error nunca equivale a «resultado correcto».

| | |
|---|---|
| Bloque 1 | Ejecutar · crear · reparar · mejorar |
| Bloque 2 | Estructurar · iterar · validar |
| **Bloque 3** | **Definir · dar contexto · elegir la vía · resolver · comprobar** |

**Las actividades cambian. El método es el mismo.**

---
# Bola extra · reto opcional con el BOE

*(Diapositivas EXTRA 1–10)*

> 🟡 **ADAPTADA** — el código de la IA se pega en una celda `%%writefile` y se ejecuta con `!python` (o `%run` si pide datos).
> La API del BOE se consulta desde servidores de Google: si falla, la referencia tiene respaldo («Solo si te has bloqueado»).

**Reto:** construye una herramienta que localice la legislación consolidada del BOE cuyo título incluye «consumidores»,
publicada en un intervalo de fechas, y genere una tabla con: fecha de publicación · identificador · rango · título ·
departamento · enlace.

*«Legislación consolidada»: textos normativos que el BOE ofrece en versión consolidada (actualizada). No es todo lo que se
publica cada día.*

**¿Por dónde empezar?** El sumario diario no sirve para esto (del 1 al 17/10/2026 ningún título contiene «consumidor»:
0 resultados no siempre es un error). La vía es la **legislación consolidada**, con búsqueda oficial por título y por
intervalo de fechas. Mismo método: ELEGIR LA VÍA antes de pedir código.

Prompt base (incluye las pistas 1 y 2 al final; míralas solo si hacen falta). Añade: *«Lo voy a ejecutar en Google Colab con `!python mi_boe.py`.»*

In [ ]:
ir_a("PROMPTS")
!cat 04_PROMPT_BOLA_EXTRA_BOE.txt

In [ ]:
ir_a("BOE")

In [ ]:
%%writefile mi_boe.py
# Pega aquí, debajo de esta línea, el código que te ha dado la IA

In [ ]:
ir_a("BOE")
!python mi_boe.py

Si tu programa te pide datos, usa esta celda:

In [ ]:
ir_a("BOE")
%run mi_boe.py

In [ ]:
# Mira tu Excel (escribe su nombre entre las comillas)
ir_a("BOE")
!ls -la *.xlsx
ver_excel("nombre_de_tu_excel.xlsx", filas=(1, 15), columnas=6)

**Comprobación** (hazla tú):
- ¿Todas las fechas de publicación están dentro del intervalo?
- ¿Todos los títulos contienen la palabra?
- ¿Hay identificadores repetidos?
- Abre dos enlaces: ¿es la norma correcta?
- ¿Aparece la Ley 4/2022 de protección de consumidores vulnerables?
- Con el intervalo 01/01/2020 – 17/10/2026, ¿obtienes 12 normas, como la referencia? *(El 12 solo sirve con ese mismo
  intervalo y palabra.)*

---
**Ya sabemos obtener la información y comprobarla.** Obtener la información y conservarla como prueba son dos problemas
distintos. ¿Cómo conservamos lo que vimos para poder revisarlo y utilizarlo como prueba? → **Bloque 4.**

---
# Solo si te has bloqueado

## Recuperación · práctica Geoportal

Script de referencia (`GEOPORTAL/script_referencia_provincia.py`).

**a) Te pregunta fechas y provincia** (aparecerá un cuadro de texto):

In [ ]:
ir_a("GEOPORTAL")
%run script_referencia_provincia.py

**b) Con las fechas en la orden** (cámbialas por las tuyas):

In [ ]:
ir_a("GEOPORTAL")
!python script_referencia_provincia.py 12-10-2026 13-10-2026 14-10-2026 --provincia 28

**c) Sin conexión con la API** (si desde Colab falla o corta): usa las respuestas reales guardadas del 12, 13 y 14/10/2026,
provincia 28. El Excel lo indica en su nombre (`_RESPALDO`).

In [ ]:
ir_a("GEOPORTAL")
!python script_referencia_provincia.py --respaldo RESPALDO_API

## Referencia · bola extra BOE

Con conexión:

In [ ]:
ir_a("BOE")
!python boe_legislacion_consumidores.py --palabra consumidores --desde 20200101 --hasta 20261017

Sin conexión (respuesta real guardada el 18/10/2026):

In [ ]:
ir_a("BOE")
!python boe_legislacion_consumidores.py --respaldo RESPALDO_BOE/consolidada_consumidores_2020_2026.json

Variante por sumario diario («qué se publicó tal día»), sin conexión:

In [ ]:
ir_a("BOE")
!python boe_buscar_disposiciones.py --respaldo RESPALDO_BOE --palabra consumo --campo todo

## Restaurar los materiales originales

**Atención:** borra también todo lo que hayas creado (`mi_geoportal.py`, `mi_boe.py`, los Excel generados y los
archivos subidos).

In [ ]:
if _servidor is not None:
    _servidor.terminate()
    _servidor = None
shutil.rmtree(RAIZ, ignore_errors=True)
with zipfile.ZipFile(ZIP_LOCAL) as z:
    z.extractall(TRABAJO)
print("Materiales restaurados.")
ir_a("")